# Hệ thống gợi ý tài liệu & khóa học CNTT
## Content-Based + TF-IDF + Cosine Similarity + Hybrid Scoring

**Hướng dẫn:** Chọn kernel Python đúng (`.venv` hoặc `IT Recommender`), sau đó **Run All**.

Nếu lỗi `No module named pandas` → chạy cell cài đặt bên dưới trước.

## 0. Cài đặt thư viện (chạy 1 lần nếu thiếu package)

In [1]:
import sys
import subprocess

REQUIRED = ["pandas", "scikit-learn", "nltk", "ipykernel"]
missing = []
for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_") if pkg != "scikit-learn" else "sklearn")
    except ImportError:
        missing.append(pkg)

if missing:
    print("Dang cai:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, "-q"])
    print("Cai dat xong! Restart kernel roi chay lai tu cell 1.")
else:
    print("Tat ca thu vien da san sang.")

Tat ca thu vien da san sang.


## 1. Import & thiết lập đường dẫn

In [2]:
import os
import sys
import pickle
from pathlib import Path

import pandas as pd

# Dam bao chay tu thu muc goc project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "it_learning_items.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from recommender_utils import (
    compute_similarity,
    create_vectorizer,
    recommend,
    tags_to_corpus,
)

print("Project root:", PROJECT_ROOT)
print("Import thanh cong!")

Project root: C:\Movie-Recommender-System-Using-Machine-Learning
Import thanh cong!


## 2. Load dataset IT

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "it_learning_items.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay: {DATA_PATH}")

items = pd.read_csv(DATA_PATH)
items.dropna(inplace=True)
print(f"So muc: {len(items)}")
print(f"Khoa hoc: {(items['type']=='Khóa học').sum()} | Tai lieu: {(items['type']=='Tài liệu').sum()}")
items.head()

So muc: 1000
Khoa hoc: 502 | Tai lieu: 498


,item_id,title,type,description,category,topics,instructor,platform,link
0,1,Giáo trình Machine Learning,Tài liệu,"Documentation đầy đủ về Machine Learning, GPT ...",Trí tuệ nhân tạo,"Machine Learning, GPT, BERT, Computer Vision",Andrew Ng,Stanford Online,https://example.com/it/giáo-trình-machine-lear...
1,2,Sách Blockchain,Tài liệu,"Giáo trình Blockchain kinh điển: lý thuyết, th...",Công nghệ mới,"Blockchain, 5G, Ethereum, Edge Computing",Vitalik Buterin,CryptoZombies,https://example.com/it/sách-blockchain
2,3,Ứng dụng Web3 cho Công nghệ mới,Khóa học,"Xây dựng ứng dụng với Web3, tích hợp Edge Comp...",Công nghệ mới,"Web3, Edge Computing, IoT, Raspberry Pi",Vitalik Buterin,CryptoZombies,https://example.com/it/ứng-dụng-web3-cho-công-...
3,4,Hướng dẫn S3 - Điện toán đám mây,Tài liệu,"Hướng dẫn chính thức S3 với reference API, tut...",Điện toán đám mây,"S3, GCP, EC2, Lambda",Lê Văn Sơn,AWS Training,https://aws.amazon.com/training/hướng-dẫn-s3--...
4,5,Học Helm,Khóa học,"Bootcamp Helm intensive: fundamentals, advance...",DevOps,"Helm, Docker, Monitoring, Prometheus",Nguyễn Thị Vân,GitHub Skills,https://skills.github.com/học-helm


## 3. Tạo weighted tags

In [4]:
corpus = tags_to_corpus(items)
print("Tags mau (co trong so):")
print(corpus[0][:300], "...")

Tags mau (co trong so):
giáo trình machine learning giáo trình machine learning giáo trình machine learning giáo trình machine learning MachineLearning GPT BERT ComputerVision MachineLearning GPT BERT ComputerVision MachineLearning GPT BERT ComputerVision MachineLearning GPT BERT ComputerVision Trítuệnhântạo Trítuệnhântạo  ...


## 4. TRAIN — TF-IDF + Cosine Similarity

In [5]:
vectorizer = create_vectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
similarity = compute_similarity(tfidf_matrix)

print("TF-IDF matrix:", tfidf_matrix.shape)
print("Similarity matrix:", similarity.shape)
print("TRAIN XONG!")

TF-IDF matrix: (1000, 7613)
Similarity matrix: (1000, 1000)
TRAIN XONG!


## 5. Test gợi ý (Hybrid Scoring)

In [6]:
item_list = items[[
    "item_id", "title", "type", "description", "category",
    "topics", "instructor", "platform", "link",
]].copy()

test_title = item_list[item_list["category"] == "Trí tuệ nhân tạo"].iloc[0]["title"]
results = recommend(item_list, similarity, test_title)

print("Goi y cho:", test_title)
pd.DataFrame(results)[["title", "type", "category", "score", "tfidf_score"]]

Goi y cho: Giáo trình Machine Learning


,title,type,category,score,tfidf_score
0,Handbook Machine Learning,Tài liệu,Trí tuệ nhân tạo,57.2,44.3
1,Official Guide Neural Network - Trí tuệ nhân tạo,Tài liệu,Trí tuệ nhân tạo,53.3,38.4
2,Học Reinforcement Learning cho Trí tuệ nhân tạo,Khóa học,Trí tuệ nhân tạo,51.9,32.1
3,Sách Machine Learning,Tài liệu,Trí tuệ nhân tạo,50.7,34.4
4,Lập trình Machine Learning,Khóa học,Trí tuệ nhân tạo,47.7,32.7


## 6. Lưu model vào artifacts/

In [7]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

pickle.dump(item_list, open(ARTIFACTS / "item_list.pkl", "wb"))
pickle.dump(similarity, open(ARTIFACTS / "similarity.pkl", "wb"))
pickle.dump(vectorizer, open(ARTIFACTS / "tfidf_vectorizer.pkl", "wb"))

print(f"Da luu {len(item_list)} items -> {ARTIFACTS}")
print("  - item_list.pkl")
print("  - similarity.pkl")
print("  - tfidf_vectorizer.pkl")
print("\nChay app: python -m streamlit run app.py")

Da luu 1000 items -> C:\Movie-Recommender-System-Using-Machine-Learning\artifacts
  - item_list.pkl
  - similarity.pkl
  - tfidf_vectorizer.pkl

Chay app: python -m streamlit run app.py
